Conor Murray


Dataset: There are two data sets for this exercise: 1) “FlightInfo_general.csv” and 2)
“FlightInfo_times.csv”. The data includes information about all flights out of 3 DC airports (BWI, DCA, IAD)
to 3 NYC airports (JFK, LGA, EWR) in January 2004. The latter data set provides information about the
scheduled and actual departure times of these flights. The former data set provides general information
about the flights as well as weather condition, etc. 

Business Objective: The goal is to build a classification model that can predict whether a flight will be
delayed or not and provide the unbiased performance measures of this model. A flight is considered
“delayed” if it departs at least 20 minutes past its scheduled departure time

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import confusion_matrix, classification_report
dfGeneral = pd.read_csv('FlightInfo_general.csv', sep=';', encoding='utf-8-sig')
dfTimes = pd.read_csv('FlightInfo_times.csv', sep=',', encoding='utf-8-sig', dtype=str)
dfGeneral['FlightID'] = dfGeneral['FlightID'].astype(str)
dfTimes.head()

,FlightID,ScheduledDeptTime,ActualDeptTime
0,1,1455,1455
1,2,1640,1640
2,3,1245,1245
3,4,1815,1709
4,5,1039,1035


In [2]:
dfGeneral = dfGeneral.drop_duplicates(subset='FlightID')
dfAll = pd.merge(dfGeneral, dfTimes, on='FlightID')
schedFixed = []
actualFixed = []
for i in range(len(dfAll)):
    s = str(dfAll['ScheduledDeptTime'].iloc[i])
    a = str(dfAll['ActualDeptTime'].iloc[i])
    while len(s) < 4:
        s = '0' + s
    while len(a) < 4:
        a = '0' + a
    schedFixed.append(s[0:2] + ':' + s[2:4])
    actualFixed.append(a[0:2] + ':' + a[2:4])
dfAll['ScheduledDeptTime'] = schedFixed
dfAll['ActualDeptTime'] = actualFixed
dfAll['scheduledFull'] = pd.to_datetime(dfAll['Date'] + ' ' + dfAll['ScheduledDeptTime'], format='%m/%d/%Y %H:%M')
dfAll['actualFull'] = pd.to_datetime(dfAll['Date'] + ' ' + dfAll['ActualDeptTime'], format='%m/%d/%Y %H:%M')
dfAll['delayAmount'] = dfAll['actualFull'] - dfAll['scheduledFull']
dfAll['Delayed'] = (dfAll['delayAmount'] > pd.Timedelta('20 min')).astype(int)
print(dfAll.shape)
dfAll.head(10)

(2201, 17)


,FlightID,Carrier,Destination,Distance,Date,FlightNumber,Origin,Weather,DayOfWeek,DayOfMonth,TailNumber,ScheduledDeptTime,ActualDeptTime,scheduledFull,actualFull,delayAmount,Delayed
0,1,OH,JFK,184,1/1/2004,5935,BWI,0,4,1,N940CA,14:55,14:55,2004-01-01 14:55:00,2004-01-01 14:55:00,0 days 00:00:00,0
1,2,DH,JFK,213,1/1/2004,6155,DCA,0,4,1,N405FJ,16:40,16:40,2004-01-01 16:40:00,2004-01-01 16:40:00,0 days 00:00:00,0
2,3,DH,LGA,229,1/1/2004,7208,IAD,0,4,1,N695BR,12:45,12:45,2004-01-01 12:45:00,2004-01-01 12:45:00,0 days 00:00:00,0
3,4,DH,LGA,229,1/1/2004,7215,IAD,0,4,1,N662BR,18:15,17:09,2004-01-01 18:15:00,2004-01-01 17:09:00,-1 days +22:54:00,0
4,5,DH,LGA,229,1/1/2004,7792,IAD,0,4,1,N698BR,10:39,10:35,2004-01-01 10:39:00,2004-01-01 10:35:00,-1 days +23:56:00,0
5,6,DH,JFK,228,1/1/2004,7800,IAD,0,4,1,N687BR,08:40,08:39,2004-01-01 08:40:00,2004-01-01 08:39:00,-1 days +23:59:00,0
6,7,DH,JFK,228,1/1/2004,7806,IAD,0,4,1,N321UE,12:40,12:43,2004-01-01 12:40:00,2004-01-01 12:43:00,0 days 00:03:00,0
7,8,DH,JFK,228,1/1/2004,7810,IAD,0,4,1,N301UE,16:45,16:44,2004-01-01 16:45:00,2004-01-01 16:44:00,-1 days +23:59:00,0
8,9,DH,JFK,228,1/1/2004,7812,IAD,0,4,1,N328UE,17:15,17:10,2004-01-01 17:15:00,2004-01-01 17:10:00,-1 days +23:55:00,0
9,10,DH,JFK,228,1/1/2004,7814,IAD,0,4,1,N685BR,21:20,21:29,2004-01-01 21:20:00,2004-01-01 21:29:00,0 days 00:09:00,0


In [3]:
timeOfDay = []
for i in range(len(dfAll)):
    hour = dfAll['scheduledFull'].iloc[i].hour
    if hour < 12:
        timeOfDay.append('Morning')
    elif hour < 18:
        timeOfDay.append('Afternoon')
    else:
        timeOfDay.append('Evening')
dfAll['TimeOfDay'] = timeOfDay
mask = dfAll['Carrier'] != 'OH'
dfAll = dfAll[mask]
dfAll = dfAll.reset_index(drop=True)
print(dfAll.shape)
dfAll.head(20)

(2171, 18)


,FlightID,Carrier,Destination,Distance,Date,FlightNumber,Origin,Weather,DayOfWeek,DayOfMonth,TailNumber,ScheduledDeptTime,ActualDeptTime,scheduledFull,actualFull,delayAmount,Delayed,TimeOfDay
0,2,DH,JFK,213,1/1/2004,6155,DCA,0,4,1,N405FJ,16:40,16:40,2004-01-01 16:40:00,2004-01-01 16:40:00,0 days 00:00:00,0,Afternoon
1,3,DH,LGA,229,1/1/2004,7208,IAD,0,4,1,N695BR,12:45,12:45,2004-01-01 12:45:00,2004-01-01 12:45:00,0 days 00:00:00,0,Afternoon
2,4,DH,LGA,229,1/1/2004,7215,IAD,0,4,1,N662BR,18:15,17:09,2004-01-01 18:15:00,2004-01-01 17:09:00,-1 days +22:54:00,0,Evening
3,5,DH,LGA,229,1/1/2004,7792,IAD,0,4,1,N698BR,10:39,10:35,2004-01-01 10:39:00,2004-01-01 10:35:00,-1 days +23:56:00,0,Morning
4,6,DH,JFK,228,1/1/2004,7800,IAD,0,4,1,N687BR,08:40,08:39,2004-01-01 08:40:00,2004-01-01 08:39:00,-1 days +23:59:00,0,Morning
5,7,DH,JFK,228,1/1/2004,7806,IAD,0,4,1,N321UE,12:40,12:43,2004-01-01 12:40:00,2004-01-01 12:43:00,0 days 00:03:00,0,Afternoon
6,8,DH,JFK,228,1/1/2004,7810,IAD,0,4,1,N301UE,16:45,16:44,2004-01-01 16:45:00,2004-01-01 16:44:00,-1 days +23:59:00,0,Afternoon
7,9,DH,JFK,228,1/1/2004,7812,IAD,0,4,1,N328UE,17:15,17:10,2004-01-01 17:15:00,2004-01-01 17:10:00,-1 days +23:55:00,0,Afternoon
8,10,DH,JFK,228,1/1/2004,7814,IAD,0,4,1,N685BR,21:20,21:29,2004-01-01 21:20:00,2004-01-01 21:29:00,0 days 00:09:00,0,Evening
9,11,DH,LGA,229,1/1/2004,7924,IAD,0,4,1,N645BR,21:20,21:14,2004-01-01 21:20:00,2004-01-01 21:14:00,-1 days +23:54:00,0,Evening


In [4]:
X = dfAll[['Carrier', 'Destination', 'Distance', 'Origin', 'Weather', 'DayOfWeek', 'DayOfMonth', 'TimeOfDay']]
X = X.copy()
X['DayOfWeek'] = X['DayOfWeek'].astype('str')
X = pd.get_dummies(X, columns=['Carrier', 'Destination', 'Origin', 'DayOfWeek', 'TimeOfDay'])
y = dfAll['Delayed']
print('X shape:', X.shape)
print('y shape:', y.shape)
X.head(13)

X shape: (2171, 26)
y shape: (2171,)


,Distance,Weather,DayOfMonth,Carrier_CO,Carrier_DH,Carrier_DL,Carrier_MQ,Carrier_RU,Carrier_UA,Carrier_US,...,DayOfWeek_1,DayOfWeek_2,DayOfWeek_3,DayOfWeek_4,DayOfWeek_5,DayOfWeek_6,DayOfWeek_7,TimeOfDay_Afternoon,TimeOfDay_Evening,TimeOfDay_Morning
0,213,0,1,False,True,False,False,False,False,False,...,False,False,False,True,False,False,False,True,False,False
1,229,0,1,False,True,False,False,False,False,False,...,False,False,False,True,False,False,False,True,False,False
2,229,0,1,False,True,False,False,False,False,False,...,False,False,False,True,False,False,False,False,True,False
3,229,0,1,False,True,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,True
4,228,0,1,False,True,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,True
5,228,0,1,False,True,False,False,False,False,False,...,False,False,False,True,False,False,False,True,False,False
6,228,0,1,False,True,False,False,False,False,False,...,False,False,False,True,False,False,False,True,False,False
7,228,0,1,False,True,False,False,False,False,False,...,False,False,False,True,False,False,False,True,False,False
8,228,0,1,False,True,False,False,False,False,False,...,False,False,False,True,False,False,False,False,True,False
9,229,0,1,False,True,False,False,False,False,False,...,False,False,False,True,False,False,False,False,True,False


In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=1)
print('X_train:', X_train.shape, ' X_test:', X_test.shape)
nnModel = MLPClassifier(random_state=1)
nnModel = nnModel.fit(X_train, y_train)
predicted_test = nnModel.predict(X_test)
holdoutAccuracy = accuracy_score(y_test, predicted_test)
holdoutPrecision = precision_score(y_test, predicted_test, pos_label=1)
holdoutRecall = recall_score(y_test, predicted_test, pos_label=1)
print('Accuracy: ', round(holdoutAccuracy, 4))
print('Precision: ', round(holdoutPrecision, 4))
print('Recall: ', round(holdoutRecall, 4))
print('Confusion Matrix:')
print(confusion_matrix(y_test, predicted_test))
print(classification_report(y_test, predicted_test))

X_train: (1519, 26)  X_test: (652, 26)
Accuracy:  0.8328
Precision:  0.6667
Recall:  0.1043
Confusion Matrix:
[[531   6]
 [103  12]]
              precision    recall  f1-score   support

           0       0.84      0.99      0.91       537
           1       0.67      0.10      0.18       115

    accuracy                           0.83       652
   macro avg       0.75      0.55      0.54       652
weighted avg       0.81      0.83      0.78       652



In [9]:
kfold = KFold(n_splits=5, shuffle=True, random_state=1)
cvModel = MLPClassifier(random_state=1)
cvAccuracy = cross_val_score(cvModel, X, y, cv=kfold, scoring='accuracy')
cvPrecision = cross_val_score(cvModel, X, y, cv=kfold, scoring='precision')
cvRecall = cross_val_score(cvModel, X, y, cv=kfold, scoring='recall')
for i in range(5):
    print('Fold', i + 1, '- Accuracy:', round(cvAccuracy[i], 4),
          ' Precision:', round(cvPrecision[i], 4),
          ' Recall:', round(cvRecall[i], 4))
print('Average Accuracy: ', round(cvAccuracy.mean(), 4))
print('Average Precision:', round(cvPrecision.mean(), 4))
print('Average Recall:   ', round(cvRecall.mean(), 4))

Fold 1 - Accuracy: 0.8299  Precision: 0.6  Recall: 0.04
Fold 2 - Accuracy: 0.8364  Precision: 1.0  Recall: 0.0658
Fold 3 - Accuracy: 0.8664  Precision: 0.5  Recall: 0.1207
Fold 4 - Accuracy: 0.8594  Precision: 0.5714  Recall: 0.127
Fold 5 - Accuracy: 0.871  Precision: 0.35  Recall: 0.14
Average Accuracy:  0.8526
Average Precision: 0.6043
Average Recall:    0.0987


In [8]:
results = pd.DataFrame({'Metric': ['Accuracy', 'Precision (Delayed=1)', 'Recall (Delayed=1)'],'Holdout (70/30)': [holdoutAccuracy, holdoutPrecision, holdoutRecall],
'5-Fold CV (Average)': [cvAccuracy.mean(), cvPrecision.mean(), cvRecall.mean()]})
results = results.round(4)
results

,Metric,Holdout (70/30),5-Fold CV (Average)
0,Accuracy,0.8328,0.8526
1,Precision (Delayed=1),0.6667,0.6043
2,Recall (Delayed=1),0.1043,0.0987
